In [33]:
# Install all required libraries, including langchain-classic for legacy components
!pip install -qU langchain langchain-groq langchain-community langchain-classic wikipedia

In [36]:
import os
from langchain_groq import ChatGroq

# 1. Set your Groq API Key
os.environ["GROQ_API_KEY"] = "your_api_key"

# 2. Initialize the Groq Chat Model (Updated to a supported model: LLaMA 3.1 8B)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

# 3. Basic LLM Call
print("--- Basic LLM Call ---")
try:
    response = llm.invoke("Explain quantum computing in one sentence.")
    # Note: ChatGroq returns an AIMessage object, so we print '.content'
    print(response.content)
except Exception as e:
    print(f"An error occurred: {e}")

--- Basic LLM Call ---
Quantum computing is a revolutionary technology that uses the principles of quantum mechanics, such as superposition and entanglement, to process information in a fundamentally different way than classical computers, allowing for exponentially faster and more complex calculations.


In [37]:
from langchain_core.prompts import PromptTemplate

# Create a dynamic template
template = "You are an expert {profession}. Explain {concept} simply."
prompt = PromptTemplate(input_variables=["profession", "concept"], template=template)

# Format the prompt with specific variables
formatted_prompt = prompt.format(profession="chef", concept="thermodynamics")

print("--- Formatted Prompt ---")
print(formatted_prompt)

--- Formatted Prompt ---
You are an expert chef. Explain thermodynamics simply.


In [38]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define components
prompt = PromptTemplate.from_template("Tell me a short joke about {topic}")
# StrOutputParser automatically extracts the string content from the AIMessage
output_parser = StrOutputParser()

# Link components using LangChain Expression Language (LCEL)
chain = prompt | llm | output_parser

print("--- Chain Execution ---")
result = chain.invoke({"topic": "programmers"})
print(result)

--- Chain Execution ---
Why do programmers prefer dark mode?

Because light attracts bugs.


In [39]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

# Initialize memory and attach it to a conversation chain
memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory)

print("--- Conversation Turn 1 ---")
turn_1 = conversation.invoke(input="Hi, my name is Alice.")
print(turn_1['response'])

print("\n--- Conversation Turn 2 ---")
# The model will remember "Alice" from the memory buffer
turn_2 = conversation.invoke(input="What is my name?")
print(turn_2['response'])

/tmp/ipykernel_3915/2919430561.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/tmp/ipykernel_3915/2919430561.py:6: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(llm=llm, memory=memory)


--- Conversation Turn 1 ---
Nice to meet you, Alice. I'm delighted to be chatting with you today. You know, did you know that the first computer bug was actually an insect? It was a moth that got stuck in the relay of the Harvard Mark II computer in 1947. The team of engineers working on the computer taped the moth to the computer log and wrote 'First actual case of bug being found' next to it.

--- Conversation Turn 2 ---
Alice! I remember you introducing yourself as Alice just a moment ago. It's lovely to confirm your name with you, and I'm happy to be chatting with you. We were just discussing that fascinating piece of computer history, weren't we?


In [43]:
from langchain_classic.agents import load_tools, initialize_agent, AgentType

# Load external tools (Wikipedia)
tools = load_tools(["wikipedia"], llm=llm)

# Initialize the agent decision-maker
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print("--- Agent Execution ---")
agent.invoke("Who won the FIFA World Cup in 2022? Look it up on Wikipedia.")

/tmp/ipykernel_3915/2361997536.py:7: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


--- Agent Execution ---


> Entering new AgentExecutor chain...
Thought: To find the winner of the FIFA World Cup in 2022, I should look up the relevant information on Wikipedia, specifically the page for the 2022 FIFA World Cup.
Action: wikipedia
Action Input: 2022 FIFA World Cup
Observation: Page: 2022 FIFA World Cup
Summary: The 2022 FIFA World Cup was the 22nd FIFA World Cup, the quadrennial world championship for national football teams organised by FIFA. It took place in Qatar from 20 November to 18 December 2022, after the country was awarded the hosting rights in 2010. It was the first World Cup to be held in the Middle East and the Arabian Peninsula, and the second in an Asian country after the 2002 tournament in South Korea and Japan.
This tournament was the last with 32 participating teams, with the number of teams being increased to 48 for the 2026 World Cup. To avoid the extremes of Qatar's hot and humid climate in summers, the event was held in November and December, beco

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kp410279eqhv99jav150fy3p` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6067, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}